# 👋 Welcome to Data Designer on Databricks!

In this notebook, we will create a high-quality synthetic dataset using **Nvidia NeMo Data Designer**. We will build a dataset representing customers, complete with realistic personal details, purchase behaviors, and LLM-generated insights.

[Link to documentation](https://nvidia-nemo.github.io/DataDesigner/0.1.5/)

**Step 0: Install & Import**
First, let's install the library and set up our environment.



In [0]:
!pip install data-designer --quiet
dbutils.library.restartPython()

In [0]:

from data_designer.essentials import *

import logging

# Mute py4j logging to only show errors/warnings
logging.getLogger("py4j").setLevel(logging.ERROR)

import os
api_key = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()


# 🚀 Step 1: Connect to the LLM

Data Designer needs a "Brain" to generate realistic content. We will configure it to use a model hosted on **Databricks Model Serving** (GPT-5).

⚠️ **CRITICAL REQUIREMENT:** To use the **Person Sampler**, you must first load the *Nemotron Personas Dataset* into your **Unity Catalog Volume**. This provides the seed data required to generate realistic identities.
👉 [**Read the Setup Guide Here**](https://nvidia-nemo.github.io/DataDesigner/0.1.5/concepts/person_sampling/#approach-2-nemotron-personas-datasets)


* **Model Provider:** We define where the model lives.
* **Data Designer:** We initialize the engine, pointing it to a Unity Catalog Volume to store temporary assets.

In [0]:
# --- Model Selection ---
# Select the model alias and ID you wish to use.
# Ensure the model_id matches the name of your deployed model in Databricks Model Serving.
model_id = "databricks-gpt-5-2"
# model_id = "databricks-gemini-2-5-flash" # Alternative example

SYSTEM_PROMPT = "/no_think"

# --- Model Provider Configuration ---
# IMPORTANT: The endpoint must be set to the base URL of your Databricks Model Serving instance.
# The API key should be your Databricks Personal Access Token (PAT).

api_url = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()

model_providers = [
    ModelProvider(
        name="databricks",
        endpoint=f"{api_url}/serving-endpoints",
        api_key=api_key, 
    )
]

# --- Data Designer Initialization ---
# CRITICAL: The 'managed_assets_path' must point to a Unity Catalog (UC) Volume.
# This volume is required to store and retrieve the NeMo personas dataset.
data_designer = DataDesigner(
    model_providers=model_providers, 
    managed_assets_path="/replace/with/your/path/to/uc/volume" # replace with your endpoint
)


# --- Configuration Builder ---
MODEL_ALIAS = "databricks"
config_builder = DataDesignerConfigBuilder(    
    model_configs=[
        ModelConfig(
            alias=MODEL_ALIAS,
            provider="databricks",
            model=model_id,
            inference_parameters=ChatCompletionInferenceParams(
                max_tokens=5000
            ),
        ),
    ]
)

# 🏗️ Step 2: Build the "Persona"

Now we start building our dataset schema.

1.  **The Anchor:** We create a `customer` column using the **Person Sampler**. This generates a rich, consistent identity (name, age, job, location) for every row.
2.  **Expression Columns:** We extract specific fields (like `city`, `email`, `occupation`) from that persona object so they appear as their own columns in our table.

In [0]:
# Persons Generator Columns

config_builder.add_column(
    name="customer",
    column_type="sampler",
    sampler_type="person",
    params={
        "locale": "en_US",
        "with_synthetic_personas": True
    },
    drop=True
)

config_builder.add_column(
    name="graph_id",
    column_type="expression",
    expr="{{customer.uuid}}"
)

config_builder.add_column(
    name="full_name",
    column_type="expression",
    expr="{{ customer.first_name }} {{ customer.last_name }}",
)

config_builder.add_column(
    name="email_address",
    column_type="expression",
    expr="{{customer.email_address}}"
)

config_builder.add_column(
    name="phone_number",
    column_type="expression",
    expr="{{customer.phone_number}}"
)


config_builder.add_column(
    name="state",
    column_type="expression",
    expr="{{customer.state}}"
)

config_builder.add_column(
    name="zipcode",
    column_type="expression",
    expr="{{customer.postcode}}"
)

config_builder.add_column(
    name="gender",
    column_type="expression",
    expr="{{customer.sex}}"
)

config_builder.add_column(
    name="age",
    column_type="expression",
    expr="{{customer.age}}"
)

config_builder.add_column(
    name="occupation",
    column_type="expression",
    expr="{{customer.occupation}}"
)

config_builder.add_column(
    name="geographic_location",
    column_type="expression",
    expr="{{customer.city}},{{customer.state}} {{customer.zipcode}}"
)

config_builder.add_column(
    name="lifestyle_segment",
    column_type="expression",
    expr="{{customer.persona}}"
)

config_builder.add_column(
    name="interests",
    column_type="expression",
    expr="{{customer.hobbies_and_interests_list}}"
)




In [0]:
for i in range(1, 4):
    config_builder.add_column(
        SamplerColumnConfig(
            name=f"_octet_{i}",
            sampler_type=SamplerType.UNIFORM,
            params=UniformSamplerParams(low=0, high=255),
            convert_to="int",
            drop=True  # Drop this column after the expression uses it
        )
    )

config_builder.add_column(
    ExpressionColumnConfig(
        name="ip_address",
        expr="{{ _octet_1 }}.{{ _octet_2 }}.{{ _octet_3 }}",        
        dtype="str"
    )
)

# 🎲 Step 3: Add Behavioral Data

Real data isn't just text; it's numbers and categories. We will add columns using statistical samplers:

* **Category:** Pick from a list (e.g., "Email" vs "Phone").
* **Poisson:** Realistic counts (e.g., "Purchases per year").
* **Gaussian:** Normal distribution (e.g., "Order history score").
* **UUID:** Unique IDs for devices or platforms.

In [0]:
# Sampler Columns

config_builder.add_column(
    SamplerColumnConfig(
        name="sensitivity",
        sampler_type=SamplerType.CATEGORY,
        params=CategorySamplerParams(
            values=[
                "Low",
                "Medium",
                "High",
            ],
            weights=[1,2,3]
        ),
    )
)

config_builder.add_column(
    SamplerColumnConfig(
        name="communication_channel",
        sampler_type=SamplerType.CATEGORY,
        params=CategorySamplerParams(
            values=[
                "Email",
                "Phone",
                "Mail",
                "Text"
            ],
        ),
    )
)



config_builder.add_column(
    SamplerColumnConfig(
        name="purchases_per_year",
        sampler_type=SamplerType.POISSON,
        params=PoissonSamplerParams(mean=5.0), # Average of 5 purchases/year
        convert_to="int"
    )
)


config_builder.add_column(
    SamplerColumnConfig(
        name="ad_receptivity",
        sampler_type=SamplerType.CATEGORY,
        params=CategorySamplerParams(
            values=[
                "Low",
                "Medium",
                "High",
            ],
        ),
    )
)

config_builder.add_column(
    SamplerColumnConfig(
        name="social_media_platform",
        sampler_type=SamplerType.CATEGORY,
        params=CategorySamplerParams(
            values=[
                "Instagram",
                "Facebook",
                "Tiktok",
                "Youtube",
                "Snapchat",
                "Other"
            ],
        ),
    )
)

config_builder.add_column(
    SamplerColumnConfig(
        name="purchase_channel_pref",
        sampler_type=SamplerType.CATEGORY,
        params=CategorySamplerParams(
            values=[
                "Mobile App",
                "Website",
                "Retail",
                "Marketplace",
                "Call Center",
                "Social Media"
            ],
        ),
    )
)


config_builder.add_column(
    SamplerColumnConfig(
        name="consent_status",
        sampler_type=SamplerType.CATEGORY,
        params=CategorySamplerParams(
            values=[
                "TRUE",
                "FALSE",
            ],
            weights=[10,1]
        ),
    )
)


config_builder.add_column(
    SamplerColumnConfig(
        name="retail_platform_id",
        sampler_type=SamplerType.UUID,
        params=UUIDSamplerParams(
            prefix="amzn_cust_",  # Optional: adds a prefix
            short_form=True,  # Optional: uses a shorter format
            uppercase=False  # Optional: uses uppercase letters
        )
    )
)

config_builder.add_column(
    SamplerColumnConfig(
        name="device_id",
        sampler_type=SamplerType.UUID,
        params=UUIDSamplerParams(
            prefix="SM",  # Optional: adds a prefix
            short_form=False,  # Optional: uses a shorter format
            uppercase=True  # Optional: uses uppercase letters
        )
    )
)


config_builder.add_column(
    SamplerColumnConfig(
        name="order_history_count",
        sampler_type=SamplerType.GAUSSIAN,
        params=GaussianSamplerParams(mean=10, stddev=2),
        convert_to="int"
    ),
)



# 🧠 Step 4: LLM Generation

This is where the magic happens! ✨

We will use **LLMTextColumnConfig** to ask the model to "think" about the customer we generated.

* *Example:* "Given this customer is a **Doctor**, what is their likely **Income**?"
* *Example:* "Given they like **Hiking**, what is their **Favorite Brand**?"

This creates synthetic data that actually makes sense contextually.

In [0]:
# LLM specific columns

# Add an LLM-generated patient message
config_builder.add_column(
    LLMTextColumnConfig(
        name="income",
        system_prompt=SYSTEM_PROMPT,
        model_alias=MODEL_ALIAS,
        prompt=(
            "{% if customer.occupation == 'not_in_workforce' %}"
            "Only respond in the following format: $0"
            "{% else %}"
            "You are estimating the income bracket for {{customer.name}} who works as : {{customer.occupation}} "
            "They live in {{customer.state}} so take into account the cost of living and make sure it is adjusted"
            "Only respond in the following format: $XXX-$YYY"
            "{% endif %}"
        ),
    )
)



config_builder.add_column(
    LLMTextColumnConfig(
        name="preferred_brand",
        system_prompt=SYSTEM_PROMPT,
        model_alias=MODEL_ALIAS,
        prompt=(
            "Give your best guess of the top brand that {{customer.name}} engages with."
            "Some information about them: {{customer.travel_persona}}, {{customer.hobbies_and_interests}}, {{customer.sports_persona}}  "
            "Only respond with the brand name."
        ),
    )
)

config_builder.add_column(
    LLMTextColumnConfig(
        name="favorite_product_category",
        system_prompt=SYSTEM_PROMPT,
        model_alias=MODEL_ALIAS,
        prompt=(
            "Give your best of {{customer.name}}'s favorite product category."
            "They typically prefer this brand : {{preferred_brand}}"
            "Their interests are : {{interests}}"
            "Only respond with the product category."
        ),
    )
)


config_builder.add_column(
    LLMTextColumnConfig(
        name="favorite_sports_teams",
        system_prompt=SYSTEM_PROMPT,
        model_alias=MODEL_ALIAS,
        prompt=(
            "Give your best guess on the favorite sports team of {{customer.name}}."
            "Some information about them: {{customer.sports_persona}} "
            "If you do not have enough information, reply with only 'none'"
            "Only respond with the sports team name"
        ),
    )
)

config_builder.add_column(
    LLMTextColumnConfig(
        name="favorite_athletes",
        system_prompt=SYSTEM_PROMPT,
        model_alias=MODEL_ALIAS,
        prompt=(
            "Give your best guess on the favorite athlete of {{customer.name}}."
            "Some information about them: {{customer.sports_persona}} "
            "If you do not have enough information, reply with only 'none'"
            "Only respond with the athlete name"
        ),
    )
)



# 👀 Step 5: Preview the Data

Before we generate millions of rows, let's run a **Preview**.

This generates a small batch (10 rows) so we can inspect the quality, check the formatting, and ensure the LLM is behaving as expected.

In [0]:
preview = data_designer.preview(config_builder)


In [0]:
preview.analysis.to_report()

In [0]:
# Convert it to a Spark DataFrame
spark_df = spark.createDataFrame(preview.dataset)

# Show the Spark DataFrame
display(spark_df)

Once the preview is to your liking, submit a batch job!

In [0]:
# job_results = data_designer.create(config_builder, num_records=20_000)